# Mechanics completion replay
Runs from an isolated environment. Software, analytical shell refinement and material-regime characterization are separate outcomes. Expected material accuracy failures are retained; completion does not mean the near-incompressible P1 formulation is accurate. No patient data or clinical qualification.


In [ ]:
import sys, os, time, pathlib, subprocess, json, hashlib, zipfile
ROOT=pathlib.Path('/content/HeartTwin_completion_'+time.strftime('%Y%m%d_%H%M%S'));ROOT.mkdir()
def checked(args,name,cwd=None,timeout=900):
    print('START',name,flush=True)
    try:
        p=subprocess.run(args,cwd=cwd,env=ENV if 'ENV' in globals() else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=timeout)
    except subprocess.TimeoutExpired as exc:
        (ROOT/name).write_text(str(exc.stdout));raise
    (ROOT/name).write_text(p.stdout);print(name,p.returncode,p.stdout[-2400:],flush=True);p.check_returncode()
checked([sys.executable,'-m','pip','install','virtualenv==21.14.12'],'bootstrap.log')
checked([sys.executable,'-m','virtualenv',str(ROOT/'venv')],'venv.log')
PY=str(ROOT/'venv/bin/python');ENV=os.environ.copy();ENV['PATH']=str(ROOT/'venv/bin')+os.pathsep+ENV['PATH'];ENV['OPENBLAS_NUM_THREADS']='1'
REPO=ROOT/'CardiMech'
checked(['git','clone','https://github.com/Virelion-Biotech/Virelion-CardiMech.git',str(REPO)],'clone.log')
checked(['git','checkout','9da8da3a395e9637f7911a10bc0c36b69b04b24a'],'checkout.log',REPO)
checked([PY,'-m','pip','install','numpy==2.1.3','scipy==1.16.3','pydantic==2.14.1','pytest==8.4.2','ruff==0.17.1','-e',str(REPO)+'[dev,validation]','git+https://github.com/Virelion-Biotech/Virelion-CardiInfer.git@9aabf87bdbee4c70b98229baa3bbea3fb896b490'],'install.log')
checked([PY,'-m','ruff','format','--check','src','tests','scripts'],'format.log',REPO)
checked([PY,'-m','ruff','check','src','tests','scripts'],'lint.log',REPO)
checked([PY,'-m','compileall','-q','src','tests','scripts'],'compile.log',REPO)
checked([PY,'-m','pytest','-q','--junitxml='+str(ROOT/'suite.xml')],'suite.log',REPO)
checked([PY,'scripts/profile_cached_operator.py','--output',str(ROOT/'operator-profile.json')],'profile.log',REPO)
checked([PY,'scripts/qualify_material_regimes.py','--output',str(ROOT/'regimes')],'regimes.log',REPO)
cases=[]
for level,layers in [(3,4),(4,6),(5,8)]:
    name=f'shell-{level}-{layers}'
    checked([PY,'scripts/validate_matrix_free.py','--level',str(level),'--layers',str(layers),'--preconditioner','block_jacobi','--output',str(ROOT/name)],name+'.log',REPO)
    cases.append(json.loads((ROOT/name/'qualification.json').read_text()))
errors=[c['relative_error_against_small_strain_Lame'] for c in cases]
assert errors[0]>errors[1]>errors[2] and errors[-1]<0.01
regimes=json.loads((ROOT/'regimes/regime-results.json').read_text())
assert len(regimes)==9 and all(r['equilibrium_passed'] for r in regimes)
checked([PY,'-m','pip','freeze'],'environment.txt',REPO)
source_paths=["src/cardimech/fem_backend.py","src/cardimech/fem_materials.py","src/cardimech/fem_krylov.py","src/cardimech/fem_linearization.py","tests/test_fem_krylov.py","tests/test_fem_linearization.py","tests/test_fem_material_regime.py","scripts/qualify_material_regimes.py","scripts/profile_cached_operator.py","scripts/validate_matrix_free.py"]
identity={p:hashlib.sha256((REPO/p).read_bytes()).hexdigest() for p in source_paths}
manifest={'source_commit':'9da8da3a395e9637f7911a10bc0c36b69b04b24a','source_sha256':identity,'recipe_executed_end_to_end':True,'software_checks_passed':True,'material_accuracy_qualification_passed':all(r['accuracy_within_15_percent'] for r in regimes),'regime_accuracy_failure_count':sum(not r['accuracy_within_15_percent'] for r in regimes),'cases':cases,'regimes':regimes,'profile':json.loads((ROOT/'operator-profile.json').read_text()),'limits':['synthetic numerical checks, no patient qualification','P1 nearly incompressible locking is NOT fixed','block preconditioning changes convergence, not discretization accuracy','timings are one Colab replay','bulky full element-field JSON omitted from archive; SHA-256 retained; all nodal coordinates/displacements retained']}
# Export raw nodal fields with the same pinned interpreter and record full-file identities.
export_code="""import pathlib,json,numpy as np,hashlib,sys
root=pathlib.Path(sys.argv[1]);records={}
for p in root.rglob('spatial-mechanics.json'):
    raw=p.read_bytes();d=json.loads(raw);out=p.with_name('nodal.npz')
    np.savez_compressed(out,reference_nodes=np.asarray(d['reference_nodes']),displacement=np.asarray(d['displacement']))
    mesh=p.with_name('mesh.json')
    records[str(p.relative_to(root))]={'sha256':hashlib.sha256(raw).hexdigest(),'bytes':len(raw),'mesh_sha256':hashlib.sha256(mesh.read_bytes()).hexdigest()}
(root/'bulk-fields-identity.json').write_text(json.dumps(records,indent=2))
"""
(ROOT/'export_nodal.py').write_text(export_code)
checked([PY,str(ROOT/'export_nodal.py'),str(ROOT)],'export.log',REPO)
(ROOT/'MANIFEST.json').write_text(json.dumps(manifest,indent=2)+'\n')
archive=ROOT/'mechanics_completion_evidence.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob('*'):
        if p.is_file() and not p.is_relative_to(ROOT/'venv') and not p.is_relative_to(REPO):
            if p.suffix in {'.log','.xml','.npz','.txt'} or p.name in {'MANIFEST.json','regime-results.json','qualification.json','operator-profile.json','bulk-fields-identity.json'}:
                z.write(p,str(p.relative_to(ROOT)))
    for p in source_paths:z.write(REPO/p,'source/'+p)
raw=archive.read_bytes()
receipt={'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest(),'git_blob':hashlib.sha1(b'blob '+str(len(raw)).encode()+b'\0'+raw).hexdigest()}
(ROOT/'receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
print('REPLAY_FINISHED',json.dumps({'root':str(ROOT),'software_checks_passed':True,'material_accuracy_qualification_passed':manifest['material_accuracy_qualification_passed'],'accuracy_failure_count':manifest['regime_accuracy_failure_count'],'receipt':receipt}),flush=True)
